# MovieLens 32M: six retrieval channels and temporal LambdaMART

This notebook has two milestones: **(1) full-Train retrieval and Validation evaluation for all six channels; (2) out-of-time internal-Train ranker examples, LambdaMART and Validation evaluation**. Test is available only through a separate, explicitly selected-artifact gate.

Only `ratings.csv`, `movies.csv`, and the frozen split notebook's derived Parquet/manifest artifacts are inputs. No tags, links, external metadata or pretrained embeddings are used. Nothing modifies external splits or the first two notebooks. The expected split counts (31,414,033 / 300,000 / 286,171) are integrity checks; matrix dimensions and training loops derive from the artifacts.

The default mode is a small synthetic **smoke_test**, which is not a benchmark. Change the mode deliberately for expensive Colab training. Every code cell is unexecuted in the distributed notebook; no performance numbers are prefilled.


In [1]:
MODE = "smoke_test"  # smoke_test | full_train | final_evaluation
RUN_RANKER_MILESTONE = False  # enable only after reviewing the complete retrieval milestone
FREEZE_AFTER_VALIDATION = False  # deliberately seal models/config after Validation review
SELECTED_RUN_ID = ""  # final_evaluation: exact ID printed by freeze_selection()
CONFIRM_TEST_RUN_ID = ""  # final_evaluation: repeat the selected ID to open the Test gate
assert MODE in {"smoke_test", "full_train", "final_evaluation"}


## Environment and resources

Use a Colab Pro L4 runtime with 24 GB VRAM. The setup cell installs **missing packages in Colab only**; it does not replace an existing CUDA PyTorch installation. Runtime package versions are recorded. No dependencies were installed locally to author this notebook. FP32 is used for both neural models; mixed precision has not been validated.

The code and tests are bundled below from `src/recsys/` for a notebook-only Colab launch. The deterministic ZIP hash identifies the source version. To edit implementations, change the repository modules and run `tools/build_notebook03.py` to regenerate this bundle. The large encoded string is packaged source, not dataset contents or model weights.


In [2]:
import importlib.util
import importlib.metadata
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Run this notebook in Google Colab. Local checks use tests/test_recsys.py.")
requirements = {
    "numpy": "numpy>=1.26", "pandas": "pandas>=2.2", "scipy": "scipy>=1.11",
    "pyarrow": "pyarrow>=15", "psutil": "psutil>=5.9", "torch": "torch>=2.2",
    "implicit": "implicit>=0.7.2", "lightgbm": "lightgbm>=4.5",
    "matplotlib": "matplotlib>=3.8", "nbformat": "nbformat>=5.10",
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print({module: importlib.metadata.version(module) for module in requirements})


{'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'pyarrow': '23.0.1', 'psutil': '5.9.5', 'torch': '2.11.0+cu130', 'implicit': '0.7.3', 'lightgbm': '4.6.0', 'matplotlib': '3.10.0', 'nbformat': '5.11.1'}


In [3]:
import base64
import hashlib
import io
import zipfile

CODE_SHA256 = "738c1eb03e05926a9e1fb49c4a471c909ccf75b78311d95e005beb079933d53d"
SOURCE_BUNDLE_B64 = "..."
_bundle = base64.b64decode(SOURCE_BUNDLE_B64)
if hashlib.sha256(_bundle).hexdigest() != CODE_SHA256:
    raise IOError("Source bundle checksum mismatch")
CODE_DIR = Path("/content/recsys_source") / CODE_SHA256
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_bundle)) as _archive:
    for _entry in _archive.infolist():
        _relative = Path(_entry.filename)
        if _relative.is_absolute() or ".." in _relative.parts or _entry.is_dir():
            raise ValueError("Unexpected source archive member")
        _target = CODE_DIR / _relative
        _payload = _archive.read(_entry)
        _target.parent.mkdir(parents=True, exist_ok=True)
        if _target.exists() and _target.read_bytes() != _payload:
            raise IOError(f"Source path already contains different code: {_target}")
        if not _target.exists():
            _target.write_bytes(_payload)
if "recsys" in sys.modules and not Path(sys.modules["recsys"].__file__).is_relative_to(CODE_DIR):
    raise RuntimeError("A different recsys source version is already loaded. Restart the Colab runtime.")
sys.path.insert(0, str(CODE_DIR))
from recsys.config import default_config
from recsys.runtime import environment
from recsys.experiment import Experiment
CONFIG = default_config()
print(environment("/content"))
print("Code SHA256:", CODE_SHA256)


{'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'gpu': {'name': 'NVIDIA L4', 'free_bytes': 23455727616, 'total_bytes': 23659151360}, 'cpu_cores': 12, 'available_ram': 54564282368, 'disk_free': 207391576064, 'packages': {'numpy': '2.1.3', 'scipy': '1.16.3', 'pandas': '2.2.3', 'pyarrow': '23.0.1', 'psutil': '5.9.5', 'torch': '2.11.0+cu130', 'implicit': '0.7.3', 'lightgbm': '4.6.0', 'matplotlib': '3.10.0', 'nbformat': '5.11.1'}}
Code SHA256: 738c1eb03e05926a9e1fb49c4a471c909ccf75b78311d95e005beb079933d53d


## Reproducible configuration and memory policy

The defaults use 64-dimensional ALS/LightGCN/Two-Tower vectors; ALS has 15 iterations, ItemCF has 100 neighbors, and all channels retrieve exactly Top-100 (or the complete remaining catalog when smaller). Full mode uses every row of Train. Resource guards can stop a stage, never silently reduce Train.

`early_stop_users` is a deterministic Validation-only tuning panel, not a training-data cap. Final Validation/Test reporting uses every eligible-positive query. Tune configuration using Validation only and keep the resulting run ID. Different code, split manifest, or configuration creates a different immutable run directory.

CPU algorithms use sparse SciPy/implicit. CUDA is reserved for neural training and chunked neural retrieval. Resource estimates are checked before CSR construction, ItemCF blocks, graph/embedding allocation, score chunks, ranker tables and Drive copies. Actual process RSS is sampled every 100 ms, and CUDA allocator peaks are recorded. Bounds are conservative estimates, not guarantees against allocator fragmentation.


In [4]:
import json
print(json.dumps(CONFIG, indent=2))
# Set configuration overrides here before creating Experiment; never change CONFIG after startup.
# Example: CONFIG["itemcf"]["block"] = 32  # smaller bounded similarity workspace
# Internal ranker tail is defined by timestamps, not by a random row sample.
# Runtime RAM/disk checks stop infeasible stages with actionable errors.


{
  "seed": 42,
  "cpu_threads": 4,
  "retrieval_k": 100,
  "query_batch": 64,
  "item_block": 4096,
  "bootstrap": 300,
  "internal_tail_days": 30,
  "unlabeled_per_query": 100,
  "feature_summary_rows": 100000,
  "early_stop_users": 2000,
  "itemcf": {
    "neighbors_per_item": 100,
    "block": 64
  },
  "als": {
    "factors": 64,
    "iterations": 15,
    "regularization": 0.05,
    "positive_confidence": 20.0,
    "low_confidence": 5.0
  },
  "lightgcn": {
    "dim": 64,
    "layers": 2,
    "epochs": 30,
    "batch_size": 8192,
    "lr": 0.01,
    "regularization": 0.001,
    "patience": 4
  },
  "two_tower": {
    "dim": 64,
    "hidden": 128,
    "epochs": 15,
    "batch_size": 4096,
    "lr": 0.001,
    "regularization": 0.0001,
    "patience": 3
  },
  "ranker": {
    "trees": 500,
    "num_leaves": 31,
    "lr": 0.05,
    "patience": 40
  }
}


## Synthetic smoke test

This exercises metric formulas, temporal boundaries, Policy B and seen filtering (including low ratings), candidate provenance, sparse ItemCF, optimized ALS when available, exact LightGCN gradients, both neural objectives, LambdaMART when available, checksummed checkpoints and sharded evaluation. It does not touch the MovieLens partitions, mount Drive, or claim full-scale results. In Colab, the package setup enables all model-library smoke tests.


In [5]:
if MODE == "smoke_test":
    import unittest
    sys.path.insert(0, str(CODE_DIR / "tests"))
    import test_recsys
    suite = unittest.defaultTestLoader.loadTestsFromModule(test_recsys)
    outcome = unittest.TextTestRunner(verbosity=2).run(suite)
    if not outcome.wasSuccessful() or outcome.skipped:
        raise RuntimeError("Smoke tests failed or skipped required model tests; review the output.")
    print("Synthetic checks passed. No full-scale training or benchmark evaluation was run.")


test_frozen_manifest_and_boundary_checks (test_recsys.CoreTests.test_frozen_manifest_and_boundary_checks) ... 

Run 5f75ef86c337074ed5ac all Train rows: 48 Policy B items: 13
ItemCF cosine rows 3/21


/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of '/usr/local/lib/python3.13/dist-packages/implicit/gpu/_cuda.so: undefined symbol: _ZN3rmm13device_bufferC1EmNS_16cuda_stream_viewENS_6detail23cccl_async_resource_refIN4cuda2mr3__412resource_refIJNS6_17device_accessibleEEEEEE'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/2 [00:00<?, ?it/s]

/content/recsys_source/738c1eb03e05926a9e1fb49c4a471c909ccf75b78311d95e005beb079933d53d/recsys/retrievers/lightgcn.py:33: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_csr_tensor(torch.as_tensor(graph.indptr.astype(np.int64), device=device),
/content/recsys_source/738c1eb03e05926a9e1fb49c4a471c909ccf75b78311d95e005beb079933d53d/recsys/retrievers/lightgcn.py:33: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cp

...

ok
test_retriever_exact_blocks_and_cold (test_recsys.CoreTests.test_retriever_exact_blocks_and_cold) ... ok
test_sharded_eval_all_channels_no_future_features (test_recsys.CoreTests.test_sharded_eval_all_channels_no_future_features) ... 

validation: 3/9 eligible users


ok
test_unknown_sampling_rejects_low_and_positive (test_recsys.CoreTests.test_unknown_sampling_rejects_low_and_positive) ... ok
test_vector_and_sparse_checkpoint_roundtrip (test_recsys.CoreTests.test_vector_and_sparse_checkpoint_roundtrip) ... ok
test_verified_checkpoint_recovery_and_corruption (test_recsys.CoreTests.test_verified_checkpoint_recovery_and_corruption) ... ok

FAIL: test_lightgbm_fit_predict (test_recsys.CoreTests.test_lightgbm_fit_predict)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/recsys_source/738c1eb03e05926a9e1fb49c4a471c909ccf75b78311d95e005beb079933d53d/tests/test_recsys.py", line 295, in test_lightgbm_fit_predict
    self.assertEqual(len(rank(m,f[f.userId==1])),10)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 7 != 10

----------------------------------------------------------------------
Ran 14 tests in 13.127s

FAILED (failures=1)


zero_hits: 1/1 eligible users
ItemCF cosine rows 3/21


RuntimeError: Smoke tests failed or skipped required model tests; review the output.

## Drive paths and immutable split validation

No dataset download is performed here: Notebook 02 has already created the frozen artifacts. Required files are `train.parquet`, `validation.parquet`, `test.parquet`, `user_eval_cohorts.parquet`, `item_eligibility.parquet`, and `split_manifest.json` under `data/splits/v1/`.

Startup streams SHA256/size checks for all manifest-listed artifacts and both raw source files. It then streams rating schemas, manifest time boundaries, counts, missing values and valid half-star ratings. This integrity inspection is not Test-based model selection. Only Validation labels enter tuning; Test model evaluation is gated later. Only Train builds histories, user/item statistics, positive edges and the candidate catalog. Cohorts are recomputed from Train positive counts using the same cold / 1–4 / 5–19 / 20+ definitions.


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/personal_projects/movielens_hybrid_recsys/")
DATASET_DIR = PROJECT_DIR / "data" / "ml-32m"
SPLIT_DIR = PROJECT_DIR / "data" / "splits" / "v1"
ARTIFACT_DIR = PROJECT_DIR / "artifacts" / "recsys_v1"
RESULT_DIR = PROJECT_DIR / "results" / "recsys_v1"
if MODE != "smoke_test":
    from google.colab import drive
    drive.mount("/content/drive")
    for _path in [DATASET_DIR, SPLIT_DIR]:
        if not _path.is_dir():
            raise FileNotFoundError(f"Missing existing dataset/split directory: {_path}. Run Notebook 02 first.")
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
experiment = None
if MODE != "smoke_test":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA L4 runtime before full-scale execution.")
    experiment = Experiment(PROJECT_DIR, CONFIG, CODE_SHA256,
                            run_id=SELECTED_RUN_ID if MODE == "final_evaluation" else None)
    print(json.dumps(experiment.audit, indent=2))
    print("Manifest boundaries:", experiment.manifest["boundaries_utc_and_unix"])
    print("Train shape:", experiment.ctx.observed.shape, "observed ratings:", experiment.ctx.observed.nnz,
          "observed positives:", experiment.ctx.positive.nnz,
          "eligible Policy B movies:", int(experiment.ctx.eligible.sum()))


## Milestone 1: full-Train six-channel retrieval

**Policy B only:** a movie needs at least one Train interaction to enter the shared catalog. All movies previously rated by the user in Train, including low ratings, are excluded. Frozen Train histories remain unchanged for Validation and Test. No future rating defines eligibility. The report counts future observed positives lost to the catalog and seen filter separately. Users with at least one eligible future positive form the primary macro-metric population; no-hit queries remain in that population.

All channels score the complete eligible unseen catalog, with deterministic movie-ID tie breaking. Each returns Top-100. Cold / no-positive-history users receive a declared positive-count Popularity fallback; Genre also falls back for an empty genre profile. Fallback query counts are reported per channel. A failed model never silently becomes a popularity channel.

1. **Popularity:** Train observed-positive counts.
2. **ItemCF:** binary positive cosine similarities, self-neighbors removed, bounded Top-100 item-neighbor index. Sparse source-item blocks bound the temporary product. Full positive-history neighbor-weight sum; no history truncation. Zero-score ties follow movie ID.
3. **ALS:** optimized sparse CPU weighted implicit ALS. Positive rating: preference 1, confidence `1 + positive_confidence`; observed low rating: preference 0, confidence `1 + low_confidence`; missing pair: preference 0, confidence 1. Low values use implicit's documented signed-confidence convention. Missing is a training proxy, not verified dislike.
4. **Genre CBF:** normalized genre multi-hot movie features; normalized sums over Train positive histories. `(no genres listed)` contributes no genre.
5. **LightGCN:** fixed positive-edge bipartite graph, symmetric degree normalization, mean of layers 0…L. BPR contrasts positive edges against uniformly drawn unknown pairs, rejecting every observed Train pair. Exact forward propagation and its exact adjoint run once per epoch, while BPR gradients accumulate in microbatches across every positive edge. **One large-batch Adam step per epoch** differs from per-minibatch SGD; tune epochs/lr on Validation. No edge-by-embedding tensor, sampled subgraph or full propagation per microbatch. FP32.
6. **Hybrid Two-Tower:** user ID + frozen historical genre profile MLP; item ID + movie genre MLP, normalized output vectors and BPR proxy loss. Train-positive pairs are contrasted with sampled unknowns; no held-out history enters the towers.

Neural early stopping uses a reproducible Validation-only user panel. All six still receive complete Validation evaluation. Full-Train and internal-snapshot artifacts are in separate namespaces. Epoch embedding/state checkpoints and completed model checkpoints are hash verified. Restart skips complete channels; an interrupted neural channel restarts deterministically from its seed (epoch state is retained for inspection, not silently resumed).

References: [implicit signed confidence](https://github.com/benfred/implicit/blob/main/implicit/cpu/als.py), [PyTorch sparse operations](https://docs.pytorch.org/docs/stable/sparse.html), [LightGBM ranking API](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRanker.html).


In [ ]:
retrieval_validation = None
if MODE == "full_train":
    retrieval_validation = experiment.retrieval_milestone()
    print("All six full-Train channels and Validation evaluation completed.")
    print("Run ID:", experiment.run_id)
    print(json.dumps(retrieval_validation["eligibility"], indent=2))
    print("Union candidate recall ceiling:", retrieval_validation["union_recall_ceiling"])


## Retrieval comparison, candidate union and ablations

Metrics are macro-averaged per eligible-positive user: Recall@20/50/100, HitRate@20/100, NDCG@20, coverage and 95% user-bootstrap intervals. Each metric's denominator is that user's eligible future positive set. Full-catalog scores are never compared to sampled-negative metrics. Cohort reports preserve no-hit users. Per-query metric Parquet files support paired follow-up analyses.

The deduplicated union has up to 600 candidates, preserving each source's score/rank/presence/fallback flag. **Union pool recall** measures every union candidate; it is not a Top-100 ranking metric. RRF uses `sum(1 / (60 + source_rank))`. Remove-one-channel RRF and pool-recall ablations use the identical query population. Reports include overlap Jaccard, unique relevant contributions, candidate-count quantiles, coverage and the fraction of Top-20 belonging to the Train-positive-popularity top decile.

Raw per-source scores are missing when that source did not retrieve the candidate; missing scores remain NaN, ranks default to 101, and explicit presence flags distinguish absence from score zero. Coverage is Top-100 for individual retrieval, Top-20 for RRF/ablations/ranker. Latency is labeled batch-amortized; hardware/time/resource evidence is saved alongside each model.


In [ ]:
import pandas as pd
from IPython.display import display

def comparison_table(summary):
    rows = []
    for name, reports in summary["metrics"].items():
        overall = next(r for r in reports if r["cohort"] == "all")
        rows.append({"model": name, **overall,
                     "catalog_coverage": summary["catalog_coverage"][name],
                     "popular_top_decile_share": summary["top_decile_popularity_share"][name]})
    return pd.DataFrame(rows).set_index("model")

if retrieval_validation is not None:
    display(comparison_table(retrieval_validation))
    display(pd.DataFrame(retrieval_validation["retrieval_timing"]).T)
    display(pd.DataFrame(retrieval_validation["fallback"]).T)
    display(pd.Series(retrieval_validation["overlap_jaccard"], name="Jaccard"))
    display(pd.Series(retrieval_validation["unique_relevant_interactions"], name="unique positives"))
    print("Union candidate count quantiles [0, .1, .5, .9, 1]:",
          retrieval_validation["union_candidate_count_quantiles"])


In [ ]:
if retrieval_validation is not None:
    import matplotlib.pyplot as plt
    _table = comparison_table(retrieval_validation)
    _channels = ["popularity", "itemcf", "als", "genre", "lightgcn", "two_tower", "rrf"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    _table.loc[_channels, ["recall20", "recall100"]].plot.bar(ax=axes[0], ylim=(0, 1))
    axes[0].set_title("Validation: identical eligible query population")
    _table.loc[["rrf"] + ["rrf_without_" + s for s in _channels[:-1]], "pool_recall"].plot.bar(ax=axes[1], ylim=(0, 1))
    axes[1].set_title("Union recall ceiling: remove-one-channel ablations")
    fig.tight_layout()
    plt.show()
    # Tables and per-query data are already persisted. Figures are derived at runtime; no fabricated outputs.


## Milestone 2: strictly out-of-time internal ranking examples

The internal cutoff is `max(Train.timestamp) - internal_tail_days * 86400`. Retriever snapshots use every Train row at or before that cutoff; future ranking labels are strictly later and remain inside the original external Train. No internal label drives snapshot fitting, early stopping or features. Snapshots use fixed configured epochs, choosing the best training-loss epoch. They have separate mappings/checkpoints/catalogs. Full-Train retrieval artifacts remain available for inference.

The same six-source Top-100 union protocol runs against this snapshot catalog. Labels are three distinct categories: observed positive (`rating >= 4`), observed low, and unobserved/unknown. LambdaMART uses relevance 1 for observed positives and weak relevance 0 for both other categories, but preserves their separate counts. It keeps every retrieved positive and observed low, and uniformly samples up to the configured unknown budget per query. No-positive or no-contrast queries cannot train a useful ranking loss; they are counted and excluded from fitting, never excluded from unconditional evaluation.

The report includes query/row counts, positive labels, low ratings, sampled unknowns, zero-positive queries, internal time boundaries, candidate recall ceiling, and candidate eligibility losses. The ranker receives only the declared cutoff feature list; no future counts, feedback-kind fields, relevance labels or timestamps are features. Tree early stopping uses an external Validation panel with full-Train retrieval features. Final Validation metrics use all eligible queries, including zero-hit queries.

Snapshot-to-full-Train feature shifts are measured before fitting via per-feature moments/quantiles and standardized mean differences, with missing fractions. These diagnostics also reflect population/time changes; they do not establish that score scales transfer without bias. Unknown subsampling changes class balance, and dropping zero-hit training queries induces selection bias. Inference always ranks the whole candidate union.


In [ ]:
ranker_validation = None
if MODE == "full_train" and RUN_RANKER_MILESTONE:
    ranker_validation = experiment.ranker_milestone()
    _model_dir = experiment.store.load("ranker/model")
    _ranker_report = json.loads((_model_dir / "training_report.json").read_text())
    print(json.dumps(_ranker_report["internal_boundary"], indent=2))
    print(json.dumps(_ranker_report["training_counts"], indent=2))
    print("Internal candidate recall ceiling:", _ranker_report["internal_candidate_recall_ceiling"])
    display(pd.DataFrame(_ranker_report["feature_distribution_mismatch"]))
    display(pd.DataFrame(ranker_validation["unconditional"]))
    display(pd.DataFrame(ranker_validation["conditional_on_pool_hit"]))


In [ ]:
if ranker_validation is not None:
    _pipeline_table = comparison_table(retrieval_validation)
    _ranking_row = next(r for r in ranker_validation["unconditional"] if r["cohort"] == "all")
    _ranking_row = {**_ranking_row,
                    "catalog_coverage": ranker_validation["catalog_coverage_top20"],
                    "popular_top_decile_share": ranker_validation["top_decile_popularity_share"]}
    _pipeline_table = pd.concat([_pipeline_table, pd.DataFrame([_ranking_row], index=["lightgbm"])])
    display(_pipeline_table)
    _pipeline_table.loc[["popularity", "itemcf", "als", "genre", "lightgcn", "two_tower", "rrf", "lightgbm"],
                        ["ndcg20", "recall20"]].plot.bar(figsize=(12, 4), ylim=(0, 1),
                                                       title="Validation pipeline: unconditional user metrics")
    plt.tight_layout()
    plt.show()


## Ranking interpretation and selection

Unconditional NDCG@20, Recall@20 and HitRate@20 retain every eligible-positive query. Conditional reports explicitly condition on at least one relevant movie in the candidate pool. Pool recall is a separate ceiling: LambdaMART cannot recover a missing candidate. Conditional scores are not comparable to unconditional scores without this qualification.

Review Validation results before setting `FREEZE_AFTER_VALIDATION=True`. That action seals the exact code/configuration/split run ID after both milestones. Use that same code bundle and configuration in `final_evaluation` mode. A changed fingerprint is rejected, and Test mode cannot call training automatically. Test results are written once under the selected run; revisiting a completed result only verifies and reuses it.


In [ ]:
if MODE == "full_train" and FREEZE_AFTER_VALIDATION:
    _selected = experiment.freeze_selection()
    print("Frozen selected run ID:", _selected)
    print("Keep this CONFIG and code bundle. Use final_evaluation with both ID fields set to this value.")


## Explicitly gated final Test evaluation

This cell loads the selected six full-Train retrievers and LambdaMART. It keeps Train-only histories fixed and does not update from Validation. The query population, Policy B catalog, metric definitions, candidate budgets and ablations match Validation. Test is never used for hyperparameter selection. Test integrity was inspected earlier; this gate controls model scoring and performance evaluation.


In [ ]:
test_summary = None
if MODE == "final_evaluation":
    if not SELECTED_RUN_ID or CONFIRM_TEST_RUN_ID != SELECTED_RUN_ID:
        raise PermissionError("Set both IDs explicitly to the same frozen selected run before opening Test.")
    test_summary = experiment.final_evaluation(CONFIRM_TEST_RUN_ID)
    print(json.dumps(test_summary["eligibility"], indent=2))
    display(comparison_table(test_summary))
    display(pd.DataFrame(test_summary["conditional_on_pool_hit"]["lightgbm"]))
    print("Union candidate recall ceiling:", test_summary["union_recall_ceiling"])


## Resource profiles, artifacts and example recommendations

Artifacts include the experiment configuration and split checksum, canonical mappings, per-channel fitted factors/neighbor index/embeddings, neural epoch weights and optimizer state, training logs, Validation metrics, candidate Parquet shards, per-query metrics and RSS/CUDA/time profiles. A matching completed stage is reused only after all recorded checksums pass. Corrupt or incomplete visible checkpoints raise an error; hidden interrupted copies are ignored and can be inspected manually. Nothing overwrites an existing valid artifact. Frequent temporary work uses local storage; completed stages are copied to Drive and verified before being marked complete.

If a channel exceeds available RAM/VRAM/disk, the stage fails explicitly and preserves earlier completed channels. Lower block sizes or use a larger runtime; do not substitute small-Train results. The ranker materializes a bounded-by-policy sampled candidate table after a dynamic guard; a long internal tail may require more CPU RAM. CPU sparse ItemCF and complete-catalog comparisons can be slow even when memory fits.


In [ ]:
if experiment is not None:
    display(experiment.artifact_inventory())
    print("Artifacts:", experiment.store.root)
    print("Results:", experiment.results.root)
    _summary = test_summary or retrieval_validation
    if _summary:
        print("Evaluation profile:", _summary["profile"])


In [ ]:
if experiment is not None and experiment.store.load("selected") is not None:
    import lightgbm as lgb
    from recsys.training import load_channels
    from recsys.candidate_fusion import fuse
    from recsys.ranking_features import features, FEATURES
    _models = load_channels(experiment.ctx, experiment.store, "full_train")
    _ranker_dir = experiment.store.load("ranker/model")
    _ranker = lgb.Booster(model_file=str(_ranker_dir / "model.txt"))
    _example_user = int(experiment.ctx.users[0])  # selected without inspecting future labels
    _outputs = {s: m.retrieve_many([_example_user], 100, CONFIG["item_block"]) for s, m in _models.items()}
    _pool = features(fuse(_outputs), experiment.ctx)
    _pool["ranking_score"] = _ranker.predict(_pool[FEATURES])
    _top20 = _pool.sort_values(["ranking_score", "movieId"], ascending=[False, True]).head(20)
    display(_top20.merge(experiment.movies, on="movieId", validate="one_to_one")
            [["movieId", "title", "genres", "ranking_score", "source_count"]])


## Scientific limitations and execution record

MovieLens ratings are observed submissions, not exposure/view logs. Submission timestamps can form rating bursts and need not match consumption time. Offline recovery of future observed high ratings measures neither online CTR nor causal satisfaction. Exposure bias, missing-not-at-random feedback, weak unknown-pair supervision and temporal changes limit interpretation. Genre-only content features miss semantic detail; low ratings remain separate from unknown pairs even when objectives share preference-zero proxies.

Policy B removes movies unseen during Train, and all future-positive exclusions are reported. Cold users use an explicit shared Popularity fallback. Candidate recall limits ranking quality. Snapshot/full-model mismatch, different internal populations and weak-label sampling can bias ranker transfer. Full-catalog scoring and exact graph propagation may cost substantial runtime on L4; memory/resource checks are preventive estimates, not a completion guarantee. Deterministic seeds are fixed, while GPU sparse kernels may still warn about nondeterminism.

**Implemented:** six full-Train retrieval channels, shared exact-catalog interface, immutable checkpoints, Validation/cohort/union/ablation metrics, out-of-time snapshot features, LambdaMART and a selected-artifact Test gate. **Local validation:** synthetic tests and static notebook/module checks only; see the repository validation report for actually exercised versus unavailable libraries. **Still required:** running full training, resource profiling, Validation selection and gated Test evaluation on Colab. No full-scale scores or execution outputs are supplied here.


In [ ]:
print("Mode:", MODE)
if experiment is None:
    print("Synthetic checks only; no MovieLens benchmark results.")
else:
    print("Run ID:", experiment.run_id)
    for _marker in ["milestone_1", "milestone_2"]:
        print(_marker, "complete" if experiment.results.load(_marker) else "not complete")
    print("Selected:", experiment.store.load("selected") is not None)
    print("Test evaluated:", experiment.results.load("final_test") is not None)
